**Simulação Computacional - Projecto Mecânico - Aula Prática 2**

---

A natureza é incrivelmente complexa e a solução de problemas reais exige o conhecimento da distribuição de *N* forças e *N* propriedades materiais que não são práticamente possíveis de calcular ponto-a-ponto, para todos os pontos de um sistema contínuo. Desta necessidade nasceu as metologias de elementos finitos que procura obter uma solução global para todo o domínio que aproxime a solução a um conjunto discreto e finito de pontos minimizando o erro dessa aproximação.  

<br>

Estamos portanto a falar genéricamente de métodos que procuram a aproximação a uma solução, onde os problemas variacionais são a base matemática que permite transformar um problema de solução difícil ou impossível em vários problemas resolúveis computacionalmente, partindo o problema em partes mais pequenas.

<br>


Para se obter uma solução numérica para um conjunto de equações diferenciais, como é o caso do método dos elementos finitos, é necessário aproximar uma qualquer função desconhecida (a solução que procuramos) através de uma soma do tipo:

$$
u(x) ≈ \sum_{i=0}^{N} c_{i} \psi_{i} (x)
$$

Onde $\psi_{i} (x)$ são funções prescritas (shape functions) e $c_{i}$ os coefiecentes desconhecidos que temos de determinar para conseguir aproximar a função desconhecida. As propostas metodológicas para a encontrar a solução aproximada têm por princípio construir N+1 equações que permitam determinar os N coeficientes, e assim podemos dividir de uma forma simplista a resolução do problema em duas fases: uma fase de construção do problema e uma outra de resolução.

<br>

Nesta aula vamos realizar a primeira implementação de FEM, no caso particular de um problema com um elementos mola unidimencionais. Usando uma abordagem por equilíbrio directo, vamos encontrar a matriz de um elemento mola linear de uma dimensão, uma mola que obedece á lei de Hooke e que resiste a forças apenas na direção do seu eixo.

---
# Introduction to the Direct Stiffness (Displacement) Method
---

<br>

## **Passo 1**: a escolha do elemento que caracteriza o problema e discretizar o problema num número finito de elementos.

<br>

Neste 1º passo a nossa escolha recai para um elemento mola que obedece à lei de Hooke, onde $F=kx$, tal que $F$ é a força exercida na mola, $k$ a constante de rigidez e $x$ uma pequena deformação quando comparada com o tamanho total da mola ($L$).

```text
     (0)                           (L)                                 
    Nodo 1 --------[ k ]--------- Nodo 2
      |                             |                             
   f1x,d1x                        f2x,d2x                          
```

<br>

## **Passo 2**: a escolha das funções de deslocamento do elemento

<br>

Aqui as funções de deslocamento da mola escolhidas são funções lineares do estilo $û = a_1 + a_2 x$, estamos aqui a fazer um salto de fé, esta função do elemento pode assumir qualquer valor no espaço internodal, mas ela tem de ser igual ao resultado da função global que queremos aproximar em cada nodo do elemento.

$$
û =
\begin{pmatrix}
1 & x \\
\end{pmatrix}
\begin{pmatrix}
a_1 \\
a_2
\end{pmatrix}
$$

Avaliando a função escolhida nos dois nodos do elemento, ficamos com:

$$
û(0) = a_1 + a_2 0 = d_{1x}  
$$

$$
û(L) = a_1 + a_2 L = d_{2x}  
$$

e portanto, resolvendo em ordem a $a_1$ e $a_2$,

$$
a1 = d_{1x}  
$$

$$
a2 = (d_{2x}-d_{1x})/L  
$$

e assim a função de deslocamento para o elemento mola ficou determinada,


$$
û(x) = d_{1x} + ((d_{2x}-d_{1x})/L)x   
$$

sendo que pode ser rescrita na forma matricial

$$
û(x) = d_{1x} + (d_{2x}x)/L-(d_{1x}x)/L   
$$

$$
û =
\begin{pmatrix}
1-x/L & x/L \\
\end{pmatrix}
\begin{pmatrix}
d_{1x} \\
d_{2x}
\end{pmatrix}
$$

Os membros da matriz $x/L$ e $1-x/L$ são chamadas as funções de forma ou funções interpoladoras.

<br>

## **Passo 3**: define as relações alongamento/deslocamento e as de stress/strain do elemento

<br>

A deformação do elemento mola $ \delta $ é dada recorrendo à função de deslocamento:

$$
\delta = û(L)-û(0) = d_{2x}-d_{1x}
$$

E a relação do deslocamento com a força é estabelecida pela lei de Hooke,

$$
T = k \delta
$$

$$
T = k (d_{2x}-d_{1x})
$$

<br>

## **Passo 4**: obtém a matriz rigidez do elemento

<br>

Para obter a matriz de rigidez do elemento basta ligar $T$ a $f_{1x}$ e $f_{2x}$,

$$
f_{1x} = -T = -1* k (d_{2x}-d_{1x}) = k(d_{1x}-d_{2x})
$$
$$
f_{2x} = T = k (d_{2x}-d_{1x})
$$

Agora recuperamos a o nosso objectivo inicial que é obter o elemento descrito na forma $F=kx$ ligando as forças aos deslocamentos para cada grau de liberdade,

$$
\begin{pmatrix}
f_{1x} \\
f_{2x} \\
\end{pmatrix}
=
\begin{pmatrix}
k & -k \\
-k & k \\
\end{pmatrix}
\begin{pmatrix}
d_{1x} \\
d_{2x} \\
\end{pmatrix}
$$

<br>
<br>


Uma vez que temos a matriz de rigidez do elemento resolvido, os próximos passos era discretizar o domínio do problema num conjunto finito de elementos, montar uma matriz de rigidez global, que contenha todas as equações de todos os elementos que compõem o modelo do problema. A esse passo segue-se a introdução de condições de fronteira que garantam que o problema tem uma solução (a matriz global não pode ser singular, ou por outras palavras, o seu determinante não pode ser zero).

Para ilustrar a montagem da matriz global e da aplicação de condições de fronteira, vamos tratar neste exercício é um problema unidimencional de duas molas em série:


```text
   (Fixo)                         (Força)->F3x                  (Força)->F2x
    Nodo 1 --------[ k1 ]-------- Nodo 3 --------[ k2 ]-------- Nodo 2
      |                             |                             |
   f1x,d1x                        f3x,d3x                       f2x,d2x  
```

Este problema está discretizado em dois elementos e 3 nodos, cada nodo tem 2 graus de liberdade (2 movimentos possíveis), notem sempre que cada elemento é sempre descrito matemáticamente nas suas coordenadas locais enquanto que a solução global necessita de coordenadas globais.

<br>

## **Passo 5**: obtém a matriz rigidez global do problema

<br>

### Direct Equilibrium Approach

No método de equilíbrio directo pegamos nas matrizes de cada um dos elementos que os compõem e temos em consideração a numeração que atribuimos a cada nodo, a cada elemento e que define a ligação entre eles.

$$
\begin{pmatrix}
f_{1x} \\
f_{3x}^{(1)} \\
\end{pmatrix}
=
\begin{pmatrix}
k_1 & -k_1 \\
-k_1 & k_1 \\
\end{pmatrix}
\begin{pmatrix}
d_{1x} \\
d_{3x}^{(1)} \\
\end{pmatrix}
$$

<br>

$$
\begin{pmatrix}
f_{3x}^{(2)} \\
f_{2x} \\
\end{pmatrix}
=
\begin{pmatrix}
k_2 & -k_2 \\
-k_2 & k_2 \\
\end{pmatrix}
\begin{pmatrix}
d_{3x} \\
d_{2x} \\
\end{pmatrix}
$$

<br>

Neste método usa-se a continuidade para definir as equações globais. Ou seja, o nodo 3 que é comum aos dois elementos garante um ponto de ligação onde:

$$
d_{3x}^{(1)} = d_{3x}^{(2)} = d_{3x}
$$

<br>

E então da mesma forma que para o deslocamento há uma continuidade, no caso das forças, um equilíbrio existe para cada nodo entre as forças internas (pares ação/reação) e as externas.

$$
F_{3x} = f_{3x}^{(1)} + f_{3x}^{(2)}
$$

$$
F_{2x} = f_{2x}^{(2)}
$$

$$
F_{1x} = f_{1x}^{(1)}
$$

<br>

Se a estas equações de forças globais introduzirmos as equações de cada elemento e a continuidade ficamos com:

$$
F_{3x} = (-k_1 d_{1x} + k_1 d_{3x}) + (k_2 d_{3x} - k_2 d_{2x})
$$

$$
F_{2x} = -k_2 d_{3x} + k_2 d_{2x}
$$

$$
F_{1x} = k_1 d_{1x} - k_1 d_{3x}
$$

<br>

E assim obtivemos a matriz global do problema através do equilíbrio de forças que completa a relação $F=Kd$ para todo o problema.

$$
\begin{pmatrix}
F_{3x} \\
F_{2x} \\
F_{1x} \\  
\end{pmatrix}
=
\begin{pmatrix}
k_1+k_2 & -k_2 & -k_1 \\
-k_2 & k_2 & 0 \\
-k_1 & 0 & k_1 \\
\end{pmatrix}
\begin{pmatrix}
d_{3x} \\
d_{2x} \\
d_{1x} \\  
\end{pmatrix}
$$

<br>

Reorganizando,

$$
\begin{pmatrix}
F_{1x} \\
F_{2x} \\
F_{3x} \\  
\end{pmatrix}
=
\begin{pmatrix}
k_1 & 0 & -k_1 \\
0 & k_2 & -k_2 \\
-k_1 & -k_2 & k_1+k_2 \\
\end{pmatrix}
\begin{pmatrix}
d_{1x} \\
d_{2x} \\
d_{3x} \\  
\end{pmatrix}
$$

<br>

$$
K =
\begin{pmatrix}
k_1 & 0 & -k_1 \\
0 & k_2 & -k_2 \\
-k_1 & -k_2 & k_1+k_2 \\
\end{pmatrix}
$$

### Superposition Method (Direct Stiffness Method)

O método de sobreposição é outra forma de obter a matriz global do problema, uma forma mais conveniente. Para o mesmo problema das duas molas temos as duas matrizes de rigidez, uma de cada elemento:

$$
k^{(1)}
=
\begin{pmatrix}
k_1 & -k_1 \\
-k_1 & k_1 \\
\end{pmatrix}
\begin{pmatrix}
d_{1x} \\
d_{3x} \\
\end{pmatrix}
$$

<br>

$$
k^{(2)}
=
\begin{pmatrix}
k_2 & -k_2 \\
-k_2 & k_2 \\
\end{pmatrix}
\begin{pmatrix}
d_{3x} \\
d_{2x} \\
\end{pmatrix}
$$

Sabemos que a matriz final é uma matriz quadrada com 3x3 (número de nós) e portanto podemos rescrever as equações de cada elemento neste formato, colocando zero onde não existir uma relação, assim para o elemento 1:

$$
k_1^{(1)}
\begin{pmatrix}
1 & 0 & -1 \\
0 & 0 & 0 \\
-1 & 0 & 1 \\
\end{pmatrix}
\begin{pmatrix}
d_{1x} \\
d_{2x} \\
d_{3x} \\
\end{pmatrix}
+
=
\begin{pmatrix}
f_{1x} \\
f_{2x} \\
f_{3x} \\
\end{pmatrix}
$$

<br>

E para o elemento 2:

$$
k_2^{(2)}
\begin{pmatrix}
0 & 0 & 0 \\
0 & 1 & -1 \\
0 & -1 & 1 \\
\end{pmatrix}
\begin{pmatrix}
d_{1x} \\
d_{2x} \\
d_{3x} \\
\end{pmatrix}
=
\begin{pmatrix}
f_{1x} \\
f_{2x} \\
f_{3x} \\
\end{pmatrix}
$$

<br>

E com os mesmo equíbrio de forças visto anteriormente:

$$
\begin{pmatrix}
f_{1x}^{(1)} \\
0 \\
f_{3x}^{(1)} \\
\end{pmatrix}
+
\begin{pmatrix}
0 \\
f_{2x}^{(2)} \\
f_{3x}^{(2)} \\
\end{pmatrix}
=
\begin{pmatrix}
F_{1x} \\
F_{2x} \\
F_{3x} \\
\end{pmatrix}
$$

<br>

$$
k_1^{(1)}
\begin{pmatrix}
1 & 0 & -1 \\
0 & 0 & 0 \\
-1 & 0 & 1 \\
\end{pmatrix}
\begin{pmatrix}
d_{1x} \\
d_{2x} \\
d_{3x} \\
\end{pmatrix}
+
k_2^{(2)}
\begin{pmatrix}
0 & 0 & 0 \\
0 & 1 & -1 \\
0 & -1 & 1 \\
\end{pmatrix}
\begin{pmatrix}
d_{1x} \\
d_{2x} \\
d_{3x} \\
\end{pmatrix}
=
\begin{pmatrix}
F_{1x} \\
F_{2x} \\
F_{3x} \\
\end{pmatrix}
$$

<br>

O que resulta numa matriz igual à que foi calculada anteriormente, mas agora por este processo.

$$
K =
\begin{pmatrix}
k_1 & 0 & -k_1 \\
0 & k_2 & -k_2 \\
-k_1 & -k_2 & k_1+k_2 \\
\end{pmatrix}
$$

<br>

<br>

## **Passo 6**: introdução de condições de fronteira e solução do problema

<br>

A introdução de condições de fronteira (cinemáticas e de suporte) tem a utilidade de tornar a matriz $K$ não singular e portanto permite obter uma uma solução. Como regra geral as condições de fronteiras necessárias para resolver o problema são do mesmo número que os movimentos possíveis pelo corpo rígido.

Das condições de fronteira do nosso problema temos que o nodo 1 é fixo, e portanto o seu deslocamento é nulo $d_{1x}=0$, e portanto o sistema de equações que define o nosso problema fica:


$$
\begin{pmatrix}
k_1 & 0 & -k_1 \\
0 & k_2 & -k_2 \\
-k_1 & -k_2 & k_1+k_2 \\
\end{pmatrix}
\begin{pmatrix}
0 \\
d_{2x} \\
d_{3x} \\
\end{pmatrix}
=
\begin{pmatrix}
F_{1x} \\
F_{2x} \\
F_{3x} \\
\end{pmatrix}
$$



<br>

Onde $F_{1x}$ é a reação desconhecida e $F_{2x}$ e $F_{3x}$ são cargas conhecidas e homogéneas. A matriz $K$ de rigidez global tem como propriedades: ser simétrica; ser singular e portanto não tem inversa até serem adicionadas as condições de fronteira suficientes para remover a singularidade e prevenir o movimento de corpo rígido; Os elementos da diagonal principal de $K$ são sempre positivos.

**Com base no sistema de equações que define o problema global e reflete as suas condições de fronteira, resolvemos o sistema para encontrar os deslocamentos desconhecidos e depois as forças desconhecidas.**

Vamos agora exemplicar a solução de sistema de três molas para ilustrar este método directo da rigidez.

<br>

### Exemplo 2.1


![Imagem:](https://drive.google.com/uc?export=view&id=119K-rAxmwRk1OOPLHXP_ZhuWHOG0Fs-_)


<br>


```text
   (Fixo)                                  (Força)              (Fixo)
                                             5N ->
    Nodo 1 ----[k1]---- Nodo 3 ----[k2]---- Nodo 4 ----[k3]---- Nodo 2
      |                   |                   |                   |
   f1x,d1x              f3x,d3x             f4x,d4x            f2x,d2x  
```

Para esta montagem de molas vamos obter (a) a matriz de rigidez global, (b) o deslocamento dos nodos 3 e 4,(c) as forças de reação nos nodos 1 e 2, e (d) as forças em cada mola. Uma força de 5N é aplicada no nodo 4 na direção do eixo dos x. As molas têm constantes de rigidez distintas, k1 = 250 N/m, k2 = 350 N/m, k3 = 450 N/m.

NOTA: O eixo dos xx local coincide com o eixo dos xx global pelo que não há lugar a transformações entre sistemas de coordenadas.

In [ ]:
import sympy as sp
from sympy import symbols, Matrix, zeros, solve, Equality
from IPython.display import display

# Define variáveis simbólicas
k1_sym, k2_sym, k3_sym = symbols('k1 k2 k3')
d1, d2, d3, d4 = symbols('d1 d2 d3 d4')
F1, F2 = symbols('F1 F2') # Forças de reação nos nós fixos

# --- (a) Montagem da Matriz de Rigidez Global ---
# Mapeamento de nós (da esquerda para a direita no diagrama):
# Nó 1 do diagrama -> d1 (fixo)
# Nó 3 do diagrama -> d3 (livre)
# Nó 4 do diagrama -> d4 (força aplicada)
# Nó 2 do diagrama -> d2 (fixo)

# Matriz de rigidez do elemento para uma única mola conectando os nós i e j
def get_element_stiffness(k_val):
    return Matrix([[k_val, -k_val], [-k_val, k_val]])

# Inicializa a matriz de rigidez global (4x4 para 4 nós)
K_global = zeros(4, 4)

# Adiciona contribuições de cada elemento de mola usando a superposição
# Nota que [i,j], onde i > linha, j>coluna
# Mola 1 (k1) conecta d1 e d3
K_global[0, 0] += get_element_stiffness(k1_sym)[0,0]
K_global[0, 2] += get_element_stiffness(k1_sym)[0,1]
K_global[2, 0] += get_element_stiffness(k1_sym)[1,0]
K_global[2, 2] += get_element_stiffness(k1_sym)[1,1]

# Mola 2 (k2) conecta d3 e d4
K_global[2, 2] += get_element_stiffness(k2_sym)[0,0]
K_global[2, 3] += get_element_stiffness(k2_sym)[0,1]
K_global[3, 2] += get_element_stiffness(k2_sym)[1,0]
K_global[3, 3] += get_element_stiffness(k2_sym)[1,1]

# Mola 3 (k3) conecta d4 e d2
K_global[3, 3] += get_element_stiffness(k3_sym)[0,0]
K_global[3, 1] += get_element_stiffness(k3_sym)[0,1]
K_global[1, 3] += get_element_stiffness(k3_sym)[1,0]
K_global[1, 1] += get_element_stiffness(k3_sym)[1,1]

print("\n(a) Matriz de Rigidez Global [K]:")
display(K_global)

#-----------------------------------------------------------------------------

# Define os Vetores Globais de Deslocamento e Força
# A força externa foi aplicada no nó 4 com o valor de 5N, e o nó 3 não tem qualquer força externa aplicada.
# Aplica as condições de fronteira
# d1 = 0 (Nó 1 fixo)
# d2 = 0 (Nó 2 fixo)

U_global = Matrix([0, 0, d3, d4])
F_global = Matrix([F1, F2, 0, 5])

print("\nVectores de deslocamento global e força global (com BCs):")
display(U_global, F_global)

# Cria o sistema de equações K * U = F
# Trabalharemos com o sistema reduzido para deslocamentos desconhecidos d3, d4 e para as forças conhecidas
# Extrai as linhas e colunas relevantes para os deslocamentos desconhecidos
K_reduced = K_global[2:4, 2:4]
U_unknown = U_global[2:4, 0]  # d3, d4
F_known = F_global[2:4, 0] # Forças conhecidas em d3 (0) e d4 (5)
print("\nSistema reduzido de equações:")
display(K_reduced,U_unknown,F_known)

# Formula as equações para d3, d4
# (NOTA: aqui é pegar na equação, mover F para o outro lado, ficando igual a zero, e é este o sistema a resolver)
equations = K_reduced * U_unknown - F_known
print("\nEquações a resolver:")
display(equations)

# Resolve para d3 e d4
solution_d = solve(equations, (d3, d4))
print("\nDeslocamentos Desconhecidos (d3 e d4):")
display(solution_d)

# Substitui os valores numéricos para as rigidezes
k_values = {k1_sym: 250, k2_sym: 350, k3_sym: 450}

d3_num = solution_d[d3].subs(k_values)
d4_num = solution_d[d4].subs(k_values)

print("\n (b) Concretização numérica do resultado (k1=250, k2=350, k3=450):")
print(f"d3 = {d3_num:.6f} m")
print(f"d4 = {d4_num:.6f} m")

#-------------------------------------------------------------------------------

# (c) Forças de Reação
# Usa o K_global * U_global = F_global original e os deslocamentos agora resolvidos
U_global = U_global.subs([(d3,d3_num),(d4,d4_num)])
K_global = K_global.subs(k_values)
F_full_calculated = K_global * U_global

# As forças de reação são os elementos correspondentes aos nós fixos (d1 e d2)
print("\n(c) Forças de Reação nos Nós 1 e 2 (F1 e F2 no nosso mapeamento):")
display(Equality(F1, F_full_calculated[0]), Equality(F2, F_full_calculated[1]))

# Verifica o equilíbrio (a soma de todas as forças deve ser zero)
# As forças externas conhecidas são 0 em d3 e 5 em d4.
print(f"\nVerificação de Equilíbrio: F1 + F2 + F_ext_d3 + F_ext_d4 = {F_full_calculated[0] + F_full_calculated[1] + 0 + 5:.6f} N")

#-------------------------------------------------------------------------------

# (d) Forças em cada mola
# Para obter as forças em cada mola
# Força na mola i = k_i * (deslocamento do nó j - deslocamento do nó i)

# Mola 1 (k1) conecta d1 e d3
F_spring1_expr = k1_sym * (d3 - d1)
F_spring1_num = F_spring1_expr.subs(k_values).subs({d3: d3_num, d1: 0})
print("\n(d) Forças em cada mola:")
print(f"Força na mola 1 (F_13) = {F_spring1_expr}")
print(f"F_13 Numérica = {F_spring1_num:.6f} N")

# Mola 2 (k2) conecta d3 e d4
F_spring2_expr = k2_sym * (d4 - d3)
F_spring2_num = F_spring2_expr.subs(k_values).subs({d4: d4_num, d3: d3_num})
print(f"Força na mola 2 (F_34) = {F_spring2_expr}")
print(f"F_34 Numérica = {F_spring2_num:.6f} N")

# Mola 3 (k3) conecta d4 e d2
F_spring3_expr = k3_sym * (d2 - d4) # d2 é o deslocamento do Nó 2 (fixo, d2=0)
F_spring3_num = F_spring3_expr.subs(k_values).subs({d2: 0, d4: d4_num})
print(f"Força na mola 3 (F_42) = {F_spring3_expr}")
print(f"F_42 Numérica = {F_spring3_num:.6f} N")


(a) Matriz de Rigidez Global [K]:


Matrix([
[ k1,   0,     -k1,       0],
[  0,  k3,       0,     -k3],
[-k1,   0, k1 + k2,     -k2],
[  0, -k3,     -k2, k2 + k3]])


Vectores de deslocamento global e força global (com BCs):


Matrix([
[ 0],
[ 0],
[d3],
[d4]])

Matrix([
[F1],
[F2],
[ 0],
[ 5]])

...

{d3: 5*k2/(k1*k2 + k1*k3 + k2*k3), d4: (5*k1 + 5*k2)/(k1*k2 + k1*k3 + k2*k3)}


 (b) Concretização numérica do resultado (k1=250, k2=350, k3=450):
d3 = 0.004895 m
d4 = 0.008392 m

(c) Forças de Reação nos Nós 1 e 2 (F1 e F2 no nosso mapeamento):


Eq(F1, -175/143)

Eq(F2, -540/143)


Verificação de Equilíbrio: F1 + F2 + F_ext_d3 + F_ext_d4 = 0.000000 N

(d) Forças em cada mola:
Força na mola 1 (F_13) = k1*(-d1 + d3)
F_13 Numérica = 1.223776 N
Força na mola 2 (F_34) = k2*(-d3 + d4)
F_34 Numérica = 1.223776 N
Força na mola 3 (F_42) = k3*(d2 - d4)
F_42 Numérica = -3.776224 N



---

## Exercícios de Consolidação

Convido-vos a resolverem o problema P2-13 da página 63 do *FEM First Course - Logan 5th Edition*.

![Imagem:](https://drive.google.com/uc?export=view&id=1n2WHauzzzwdtrHTlbIzRuqiufWQ12124)

Determina os deslocamentos nodais, as forças e as reações em cada elemento.

---

<br>


